# Track 1 · 14 EEGNet-style network, two-stage training, on Alljoined

**One job.** Train a small EEG-shaped network (temporal filters, then one spatial filter over all electrodes)
against the same DINOv2-giant targets, with the same held-out and validation people as track1_11/13, and see
whether it beats chance on unseen people. Needs `track1_12_alljoined_prepare` to have run. Self-contained.

Why this model: the ImageNet ResNet in track1_13 never left chance (train loss 5.5568 to 5.5454, chance 5.5452;
held-out people 2.62% / 2.20% top-5, chance 2.50%). An image CNN treats electrodes as pixel rows; EEGNet is built for EEG.

## The network (`solvers/track1_image_eegnet2.py`, plain torch)

Window (32 electrodes x 120 samples), standardised per window, then temporal conv (16 filters, 1x64), depthwise
spatial conv over all 32 electrodes (D 2 = 32 maps), AvgPool 4, separable conv to F2 = 64 maps, AvgPool 2, and a
funnel head (Linear to 512, GELU, dropout, Linear to 1536). The feature map entering the head is 64 x 15 = **960**
values (the stock braindecode EEGNet has 16 x 3 = 48, which confines the 1536-d output to a 48-dimensional subspace).
About 1.28M parameters.

## Two-stage training (the user's design)

1. **Stage 1** (`STAGE1_EPOCHS`, default 3): all weights train, dropout 0.5, lr 2e-3: a broad, regularised first fit.
2. **Stage 2** (`STAGE2_MODE`): dropout 0.25, lr 5e-4, with the stage-1 weights as the starting point:
   - `'lora'` (default): base weights frozen, low-rank (LoRA) adapters on the temporal conv, the pointwise conv and
     the two head layers; BatchNorm keeps training. Merged into plain weights before export.
   - `'full'`: all weights keep training at the lower lr (control for LoRA).
   - `'none'`: stop after stage 1 (control).
   The depthwise convolutions (spatial and separable) have no adapter: LoRA is only wired for ordinary convs and
   linear layers here.
Best validation epoch over both stages is kept (saved to Drive as it improves); early stopping (patience 5) applies in stage 2.
Validation is on whole unseen people. A learning check is printed after epoch 2: if training loss is still at chance, stop.

## Views per window (as track1_13)

original | time-shifted + noisy | electrode-rows shuffled with a few electrodes zeroed. Mean loss.


## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'
DATA = 'alljoined'               # Track1/<DATA>/ written by track1_12_alljoined_prepare

N_HOLDOUT_PEOPLE = 5             # whole people never fitted or validated; scored at the end (same 5 as track1_11/13)
N_VAL_PEOPLE = 3                 # whole people kept out of fitting to choose the epoch and stop early on
HOLDOUT_PEOPLE = []              # or name them (subject ids as in the file names, e.g. ['07', '08'])

# --- the network (960 features into the head with these values) ---
F1, D, F2 = 16, 2, 64            # temporal filters, spatial filters per temporal filter, separable maps
KERNEL, POOL1, POOL2 = 64, 4, 2  # temporal kernel (samples), pool sizes
HIDDEN = 512                     # funnel head width

# --- stage 1: everything trains, heavier dropout ---
STAGE1_EPOCHS = 3
DROPOUT_1 = 0.5
LR_1 = 2e-3

# --- stage 2: continue from stage 1 with lower dropout ---
STAGE2_MODE = 'lora'             # 'lora' (adapters, base frozen) | 'full' (all weights, low lr) | 'none'
STAGE2_MAX_EPOCHS = 20           # upper bound; early stopping decides
DROPOUT_2 = 0.25
LR_2 = 5e-4
LORA_RANK, LORA_ALPHA = 8, 8     # on the temporal conv, the pointwise conv and the two head layers

WEIGHT_DECAY = 0.01
BATCH = 256
TEMPERATURE = 0.07
USE_AMP = True                   # mixed precision on GPU
MAX_FIT_WINDOWS = None           # or a number: fit on a random subset (quick checks)

AUGMENT = True                   # 3 views per window (False: originals only)
NOISE_FRAC = 0.1
SHIFT_SAMPLES = 3                # +-3 samples (25 ms) in the noisy view
MAX_DROP_ELECTRODES = 2

KEEP = 'best'                    # 'best' validation-rank epoch (saved to Drive as it happens), or 'last'
PATIENCE = 5                     # stage 2: epochs in a row without a validation-rank gain
MIN_DELTA = 0.0005
DIVERGE_LOSS = 0.15              # stage 2: val loss above its minimum by this, 2 epochs in a row

MAKE_ZIP = True
VAL_IMAGE_FRAC = 0.2
SEED = 33

ARCH_CFG = dict(F1=F1, D=D, F2=F2, kernel=KERNEL, pool1=POOL1, pool2=POOL2, hidden=HIDDEN)
RUN_TAG = (f'eegnet2_f{F2}_h{HIDDEN}_s1x{STAGE1_EPOCHS}d{DROPOUT_1}_{STAGE2_MODE}'
           + (f'r{LORA_RANK}' if STAGE2_MODE == 'lora' else '') + f'd{DROPOUT_2}_b{BATCH}'
           + (f'_val{N_VAL_PEOPLE}p' if N_VAL_PEOPLE else '')
           + (f'_fit{MAX_FIT_WINDOWS}' if MAX_FIT_WINDOWS else '')
           + (f'_aug{NOISE_FRAC:g}' if AUGMENT else '_noaug') + f'_{KEEP}')
RUN_TRAINING = True
print(f'NOTEBOOK VERSION: track1_14 v1   settings {RUN_TAG}   '
      f'chance loss will be {__import__("math").log(BATCH):.4f}')


In [ ]:
import importlib
for mod in ('torch',):
    print(f'{mod:12} {getattr(importlib.import_module(mod), "__version__", "ok")}')


In [ ]:
import json, os, shutil, sys, time, types, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
AJ = BASE / DATA
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', '/content/hf')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = USE_AMP and DEVICE == 'cuda'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def l2(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def _truth(y_ids):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    return gal, np.array([pos[y] for y in y_ids])


def rank_pct(pred, y_ids, emb):
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    return float((sims < sims[np.arange(len(sims)), truth][:, None]).mean())


def top5(pred, y_ids, emb):
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    k = min(5, len(gal))
    tk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, tk)]), 100.0 * k / len(gal)


def rank_null(pred, y_ids, n=200):
    """Validation rank under permuted labels, for these exact predictions."""
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    rows = np.arange(len(sims))
    return np.array([float((sims < sims[rows, p][:, None]).mean())
                     for p in (rng.permutation(truth) for _ in range(n))])


print(f'device {DEVICE}, mixed precision {AMP}')

## Step 1 — the Alljoined windows and targets

In [ ]:
t = np.load(AJ / 'image_embeddings.npz', allow_pickle=True)
emb = l2(t['emb'].astype('float32'))
DIM = emb.shape[1]
key = {k: i for i, k in enumerate(t['keys'].tolist())}

train_files = sorted((AJ / 'windows').glob('*_train.npz'))
test_files = sorted((AJ / 'windows').glob('*_test.npz'))
if not train_files or not test_files:
    raise FileNotFoundError(f'no windows in {AJ / "windows"}; run track1_12_alljoined_prepare first')

# Pass 1: labels, people and which windows each file contributes (small).
plan, ys, subs, CH = [], [], [], None
for f in train_files:
    z = np.load(f, allow_pickle=True)
    idx = np.array([key.get(k_, -1) for k_ in z['keys'].tolist()])
    keep = idx >= 0
    if CH is None:
        CH = list(z['channels'])
    elif list(z['channels']) != CH:
        raise RuntimeError(f'{f.name}: channel order differs from the first training file')
    plan.append((f, keep))
    ys.append(idx[keep])
    subs.append(np.full(int(keep.sum()), str(z['subject'])))
n_train = sum(int(k.sum()) for _, k in plan)

# Pass 2: the windows themselves, written once into one array on the machine's local disk
# (not Drive) and read per batch; a rerun in the same session reuses it.
local = Path('/content') if Path('/content').exists() else AJ
xfile, sigfile = local / 'track1_aj_train_X.npy', local / 'track1_aj_train_X.json'
signature = json.dumps([[str(f), f.stat().st_size, int(k.sum())] for f, k in plan])
if xfile.exists() and sigfile.exists() and sigfile.read_text() == signature:
    Xtr = np.load(xfile, mmap_mode='r')
    print(f'reusing {xfile} ({xfile.stat().st_size / 1e9:.1f} GB)')
else:
    Xtr, i, t0 = None, 0, time.time()
    for f, keep in plan:
        x = np.load(f)['X'][keep]
        if Xtr is None:
            Xtr = np.lib.format.open_memmap(xfile, mode='w+', dtype='float16',
                                            shape=(n_train,) + x.shape[1:])
        Xtr[i:i + len(x)] = x
        i += len(x)
        del x
    Xtr.flush()
    sigfile.write_text(signature)
    Xtr = np.load(xfile, mmap_mode='r')
    print(f'{n_train:,} training windows from {len(plan)} files written to {xfile} '
          f'in {time.time() - t0:.0f}s')
Wtr = [Xtr, np.concatenate(ys), np.concatenate(subs)]

parts = [[], [], []]
for f in test_files:
    z = np.load(f, allow_pickle=True)
    idx = np.array([key.get(k_, -1) for k_ in z['keys'].tolist()])
    keep = idx >= 0
    parts[0].append(z['X'][keep]); parts[1].append(idx[keep])
    parts[2].append(np.full(int(keep.sum()), str(z['subject'])))
    if list(z['channels']) != CH:
        raise RuntimeError(f'{f.name}: test channel order differs from training')
Wte = [np.concatenate(a) for a in parts]
IN_SFREQ = float(json.loads((AJ / 'manifest.json').read_text())['sfreq'])

people = sorted(set(Wtr[2].tolist()))
if not HOLDOUT_PEOPLE:
    HOLDOUT_PEOPLE = sorted(rng.choice(people, N_HOLDOUT_PEOPLE, replace=False).tolist())
HOLDOUT_PEOPLE = [str(p_) for p_ in HOLDOUT_PEOPLE]
assert set(HOLDOUT_PEOPLE) <= set(people), f'unknown people {set(HOLDOUT_PEOPLE) - set(people)}'
SEEN = ~np.isin(Wtr[2], HOLDOUT_PEOPLE)        # rows allowed into fit AND val
seen_imgs = np.unique(Wtr[1][SEEN])
val_imgs = set(rng.choice(seen_imgs, int(len(seen_imgs) * VAL_IMAGE_FRAC), replace=False).tolist())
IV = np.array([y in val_imgs for y in Wtr[1]])
# Choose the epoch on whole people who are not fitted (unseen people, unseen images): the point of
# this run is generalisation to new people, and seen people's new images would not measure that.
others = [p_ for p_ in people if p_ not in HOLDOUT_PEOPLE]
VAL_PEOPLE = sorted(rng.choice(others, N_VAL_PEOPLE, replace=False).tolist()) if N_VAL_PEOPLE else []
VALP = np.isin(Wtr[2], VAL_PEOPLE)
FIT = ~IV & SEEN & ~VALP
VAL = (IV & VALP) if N_VAL_PEOPLE else (IV & SEEN)
print(f'{len(people)} people; HELD OUT {HOLDOUT_PEOPLE}: {(~SEEN).sum():,} windows never fitted or validated')
print(f'train {Wtr[0].shape}   test {Wte[0].shape}   {len(CH)} channels at {IN_SFREQ:g} Hz')
print(f'VALIDATION people {VAL_PEOPLE or "(none: seen people, new images)"}')
print(f'{FIT.sum():,} fit / {VAL.sum():,} val windows, {len(np.unique(Wte[1]))} test images')


## Step 2 — the model, from the submission file

The submission file is executed here to get the network classes, so what is trained is exactly what is submitted.

In [ ]:
# Source of truth: solvers/track1_image_reve_lora_perchan.py in the repo.
SUBMISSION_PY = '"""Track 01 submission: a wider EEGNet-style network with a funnel head, plain torch.\n\nThe window (electrodes x time, 120 Hz) is standardised per window, then:\ntemporal convolution (one 1x64 filter bank per electrode), a depthwise spatial convolution over\nall electrodes (the step that mixes electrodes), pooling, a separable convolution, pooling, and a\nfunnel head (Linear to `hidden`, GELU, dropout, Linear to the 1536-d target). With the default\nsizes (F1 16, D 2, F2 64, pools 4 and 2) the feature map entering the head has 64 x 15 = 960 values,\ninstead of the 48 of the stock braindecode EEGNet, so the 1536-d output is not confined to a\n48-dimensional subspace.\n\nWritten in plain torch (no braindecode), so nothing is installed at evaluation. Inputs are matched to\nthe trained electrodes by name (case-insensitive); extra electrodes are ignored and missing ones are\nzero-filled. The window is trimmed or zero-padded to the trained length.\n\nThis file is also the model definition the training notebook uses, so the two cannot drift apart.\n"""\n\nimport json\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\n\nTARGET_SFREQ = 120.0\nCLIP_SD = 15.0\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum (pure torch; nothing is installed at evaluation)."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device)\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass EEGNetCLIP(nn.Module):\n    def __init__(self, channels, n_outputs, in_sfreq, F1=16, D=2, F2=64, kernel=64, depth_kernel=16,\n                 pool1=4, pool2=2, hidden=512, dropout=0.25, n_times=120):\n        super().__init__()\n        self.in_sfreq, self.n_times = float(in_sfreq), int(n_times)\n        C = len(channels)\n        self.pad_t = nn.ZeroPad2d((kernel // 2 - 1, kernel // 2, 0, 0))     # \'same\' length for an even kernel\n        self.temporal = nn.Conv2d(1, F1, (1, kernel), bias=False)\n        self.bn1 = nn.BatchNorm2d(F1)\n        self.spatial = nn.Conv2d(F1, F1 * D, (C, 1), groups=F1, bias=False)  # mixes all electrodes\n        self.bn2 = nn.BatchNorm2d(F1 * D)\n        self.pool1 = nn.AvgPool2d((1, pool1))\n        self.drop1 = nn.Dropout(dropout)\n        self.pad_s = nn.ZeroPad2d((depth_kernel // 2 - 1, depth_kernel // 2, 0, 0))\n        self.depth = nn.Conv2d(F1 * D, F1 * D, (1, depth_kernel), groups=F1 * D, bias=False)\n        self.point = nn.Conv2d(F1 * D, F2, 1, bias=False)\n        self.bn3 = nn.BatchNorm2d(F2)\n        self.pool2 = nn.AvgPool2d((1, pool2))\n        self.drop2 = nn.Dropout(dropout)\n        self.feat_dim = F2 * ((self.n_times // pool1) // pool2)\n        self.head = nn.Sequential(nn.Linear(self.feat_dim, hidden), nn.GELU(), nn.Dropout(dropout),\n                                  nn.Linear(hidden, n_outputs))\n\n    def forward(self, X):                          # (B, n_channels in trained order, T)\n        X = X.float()\n        n_out = max(int(round(X.shape[-1] * TARGET_SFREQ / self.in_sfreq)), 8)\n        X = resample_fft(X, n_out)\n        if X.shape[-1] > self.n_times:\n            X = X[..., : self.n_times]\n        elif X.shape[-1] < self.n_times:\n            X = F.pad(X, (0, self.n_times - X.shape[-1]))\n        X = (X - X.mean(dim=(1, 2), keepdim=True)) / (X.std(dim=(1, 2), keepdim=True) + 1e-8)\n        X = torch.clamp(X, -CLIP_SD, CLIP_SD)\n        x = self.bn1(self.temporal(self.pad_t(X[:, None])))\n        x = self.drop1(self.pool1(F.elu(self.bn2(self.spatial(x)))))\n        x = self.bn3(self.point(self.depth(self.pad_s(x))))\n        x = self.drop2(self.pool2(F.elu(x)))\n        return self.head(x.flatten(1))\n\n\nclass NameMatched:\n    """Reorders incoming electrodes into the trained order, zero-filling gaps."""\n\n    def __init__(self, model, channels, in_names, device):\n        self.model, self.device = model, device\n        lookup = {n.casefold(): i for i, n in enumerate(in_names)}\n        self.src = [lookup.get(c.casefold(), -1) for c in channels]\n        missing = [c for c, i in zip(channels, self.src) if i < 0]\n        print(f"[EEGNet2] matched {len(channels) - len(missing)}/{len(channels)} training channels by "\n              f"name among {len(in_names)} inputs; zero-filled: {missing if missing else \'none\'}")\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        self.model.eval()\n        return self.model(out)\n\n\nclass Solver(CompetSolver):\n\n    name = "EEGNet2-CLIP-Alljoined"\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        cfg = json.loads((sub / "eegnet_config.json").read_text())\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        model = EEGNetCLIP(cfg["channels"], cfg["n_outputs"], meta["sfreq"], **cfg["arch"])\n        state = torch.load(sub / "weights.pt", map_location=meta["device"], weights_only=True)\n        missing, unexpected = model.load_state_dict(state, strict=False)\n        if unexpected or missing:\n            raise RuntimeError(f"weights mismatch: missing {missing[:4]}, unexpected {unexpected[:4]}")\n        return NameMatched(model.to(meta["device"]).eval(), cfg["channels"], in_names, meta["device"])\n'

stub = types.ModuleType('benchmark_utils.base_solver')
stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
sys.modules['benchmark_utils.base_solver'] = stub
ns = {}
exec(compile(SUBMISSION_PY, 'submission.py', 'exec'), ns)


In [ ]:
import torch.nn as nn

class LoRAConv2d(nn.Module):
    """Frozen conv plus a rank-r update (a conv with the same kernel to r channels, then a 1x1 back).
    B starts at zero; the update merges exactly into the kernel."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Conv2d(base.in_channels, r, base.kernel_size, stride=base.stride,
                           padding=base.padding, bias=False)
        self.B = nn.Conv2d(r, base.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=5 ** 0.5)
        nn.init.zeros_(self.B.weight)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x))

    def merged_weight(self):
        delta = torch.einsum('or,rikl->oikl', self.B.weight[:, :, 0, 0], self.A.weight)
        return self.base.weight + self.scale * delta


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Linear(base.in_features, r, bias=False)
        self.B = nn.Linear(r, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=5 ** 0.5)
        nn.init.zeros_(self.B.weight)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x))

    def merged_weight(self):
        return self.base.weight + self.scale * (self.B.weight @ self.A.weight)


def add_lora(model, r, alpha):
    """Adapters on the temporal conv, the pointwise conv and the two head linears (not the depthwise
    convs, which have groups). Returns the number wrapped."""
    model.temporal = LoRAConv2d(model.temporal, r, alpha)
    model.point = LoRAConv2d(model.point, r, alpha)
    model.head[0] = LoRALinear(model.head[0], r, alpha)
    model.head[3] = LoRALinear(model.head[3], r, alpha)
    return 4


def to_plain(model):
    """Copy of the model with every adapter merged into one plain layer."""
    plain = copy.deepcopy(model)
    def walk(m):
        for name, child in list(m.named_children()):
            if isinstance(child, (LoRAConv2d, LoRALinear)):
                layer = child.base
                layer.weight = nn.Parameter(child.merged_weight().detach().clone(), requires_grad=False)
                setattr(m, name, layer)
            else:
                walk(child)
    walk(plain)
    return plain


def set_dropout(m, p):
    for mod in m.modules():
        if isinstance(mod, nn.Dropout):
            mod.p = p


import copy
if RUN_TRAINING:
    model = ns['EEGNetCLIP'](CH, DIM, IN_SFREQ, dropout=DROPOUT_1, **ARCH_CFG).to(DEVICE)
    n_all = sum(p.numel() for p in model.parameters())
    print(f'EEGNet-style network: {n_all / 1e6:.2f}M parameters, {model.feat_dim} features into the head')
    print('forward check', tuple(model.eval()(torch.randn(2, len(CH), 120, device=DEVICE)).shape),
          f'(expected (2, {DIM}))')


## Step 3 — train (stage 1, then stage 2)

Epoch-1 time and projected total print within the first 100 batches; a learning check prints after epoch 2.

In [ ]:
if RUN_TRAINING:
    EMB_T = torch.tensor(emb, device=DEVICE)


    def clip_loss(pred, idx):
        logits = (F.normalize(pred.float(), dim=-1)
                  @ F.normalize(EMB_T[idx], dim=-1).T) / TEMPERATURE
        same = idx[:, None] == idx[None, :]
        same.fill_diagonal_(False)
        logits = logits.masked_fill(same, -1e4)
        return F.cross_entropy(logits, torch.arange(len(pred), device=pred.device))


    def time_shift(x):
        B, C, T = x.shape
        sh = torch.randint(-SHIFT_SAMPLES, SHIFT_SAMPLES + 1, (B, 1, 1), device=x.device)
        idx = (torch.arange(T, device=x.device)[None, None, :] - sh).clamp(0, T - 1).expand(B, C, T)
        return x.gather(2, idx)


    def drop_electrodes(x):
        B, C, _ = x.shape
        k = torch.randint(0, MAX_DROP_ELECTRODES + 1, (B, 1), device=x.device)
        keep = torch.argsort(torch.rand(B, C, device=x.device), dim=1) >= k      # k random rows dropped
        return x * keep[:, :, None]


    def views(x):
        """The window as it is; a time-shifted noisy copy; a copy with the electrode rows shuffled and a few
        electrodes zeroed (time inside a row untouched). Originals only when AUGMENT is False."""
        out = [x]
        if AUGMENT:
            t = time_shift(x)
            out.append(t + NOISE_FRAC * t.std(-1, keepdim=True) * torch.randn_like(t))
            order = torch.argsort(torch.rand(x.shape[0], x.shape[1], device=x.device), dim=1)
            out.append(drop_electrodes(x.gather(1, order[:, :, None].expand_as(x))))
        return out


    @torch.no_grad()
    def infer(X, bs=256):
        model.eval()
        out = []
        for i in range(0, len(X), bs):
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                out.append(model(torch.from_numpy(X[i:i + bs]).to(DEVICE)).float())
        return torch.cat(out)


    def plain_state():
        return {k: v.detach().cpu().clone() for k, v in to_plain(model).state_dict().items()}


    fit_idx, val_idx = np.where(FIT)[0], np.where(VAL)[0]
    if MAX_FIT_WINDOWS and len(fit_idx) > MAX_FIT_WINDOWS:
        fit_idx = np.sort(rng.choice(fit_idx, MAX_FIT_WINDOWS, replace=False))
        print(f'fitting on a random {MAX_FIT_WINDOWS:,} of the training windows (MAX_FIT_WINDOWS)')
    Xv, yv = Wtr[0][val_idx], Wtr[1][val_idx]     # validation is read into RAM once
    yv_t = torch.tensor(yv.astype('int64'), device=DEVICE)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    print(f'chance loss {np.log(BATCH):.4f}')

    phases = [('stage 1', STAGE1_EPOCHS, DROPOUT_1)]
    if STAGE2_MODE != 'none':
        phases.append(('stage 2', STAGE2_MAX_EPOCHS, DROPOUT_2))
    best, best_state, last_state, hist = -1, None, None, []
    bad, loss_up, vl_best, ep, stopped = 0, 0, 1e9, 0, False
    for phase, n_ep, p_drop in phases:
        set_dropout(model, p_drop)
        if phase == 'stage 1':
            params, lr = list(model.parameters()), LR_1
        else:
            if STAGE2_MODE == 'lora':
                for p in model.parameters():
                    p.requires_grad_(False)
                add_lora(model, LORA_RANK, LORA_ALPHA)
                model.to(DEVICE)
                for m_ in model.modules():                  # BatchNorm keeps adapting
                    if isinstance(m_, nn.BatchNorm2d):
                        for p in m_.parameters():
                            p.requires_grad_(True)
                params = [p for p in model.parameters() if p.requires_grad]
                print(f'stage 2: LoRA on 4 layers; trainable {sum(p.numel() for p in params) / 1e6:.2f}M '
                      f'of {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')
            else:
                params = list(model.parameters())
                print('stage 2: all weights, lower learning rate')
            lr = LR_2
            bad, loss_up, vl_best = 0, 0, min([h[2] for h in hist] or [1e9])
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
        print(f'--- {phase}: up to {n_ep} epochs, dropout {p_drop}, lr {lr:g}')
        for _ in range(n_ep):
            ep += 1
            model.train()
            tot, t0 = 0.0, time.time()
            perm = rng.permutation(len(fit_idx))
            batches = [fit_idx[perm[i:i + BATCH]] for i in range(0, len(perm) - 7, BATCH)]
            for bi, rows in enumerate(batches, 1):
                if bi in (20, 100) or bi % 500 == 0:
                    print(f'    batch {bi}/{len(batches)}  {time.time() - t0:.0f}s  projected epoch '
                          f'{(time.time() - t0) / bi * len(batches) / 60:.1f} min')
                b = np.sort(rows)
                xb = torch.from_numpy(Wtr[0][b]).to(DEVICE).float()
                yb = torch.from_numpy(Wtr[1][b].astype('int64')).to(DEVICE)
                loss = 0.0
                for v in views(xb):
                    with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                        pred = model(v)
                    loss = loss + clip_loss(pred, yb)
                loss = loss / (3 if AUGMENT else 1)
                opt.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                tot += float(loss) * len(b)
            pv = infer(Xv)
            vl = float(np.mean([float(clip_loss(pv[i:i + BATCH], yv_t[i:i + BATCH]))
                                for i in range(0, len(pv) - 7, BATCH)]))
            r = rank_pct(pv.cpu().numpy(), yv, emb)
            hist.append((ep, tot / len(fit_idx), vl, r))
            last_state = plain_state()
            improved = r > best + MIN_DELTA
            if r > best:
                best, best_state = r, last_state
                torch.save(best_state, WEIGHTS / f'{RUN_TAG}_best_so_far.pt')   # survives a Colab disconnect
            if ep == 2:
                print('  learning check: ' + ('training loss is below chance, it is learning'
                      if tot / len(fit_idx) < np.log(BATCH) - 0.02 else
                      f'WARNING training loss is still at chance ({np.log(BATCH):.4f}): the model is not learning; stop and loosen'))
            if ep == 1:
                print(f'  epoch 1 took {time.time() - t0:.0f}s; about {(time.time() - t0) * (STAGE1_EPOCHS + STAGE2_MAX_EPOCHS) / 60:.0f} min if it ran all '
                      f'{STAGE1_EPOCHS + STAGE2_MAX_EPOCHS} epochs')
            print(f'  epoch {ep:2d} ({phase})  train loss {tot / len(fit_idx):.4f}  val loss {vl:.4f}  '
                  f'gap {vl - tot / len(fit_idx):+.3f}  val rank {r:.4f}{"  *" if r == best else ""}  '
                  f'{time.time() - t0:.0f}s')
            if phase == 'stage 2':
                vl_best = min(vl_best, vl)
                bad = 0 if improved else bad + 1
                loss_up = loss_up + 1 if vl > vl_best + DIVERGE_LOSS else 0
                if bad >= PATIENCE or loss_up >= 2:
                    why = (f'no validation-rank gain for {bad} epochs' if bad >= PATIENCE else
                           f'val loss {vl:.4f} is above its minimum {vl_best:.4f} by more than {DIVERGE_LOSS}')
                    print(f'  early stop at epoch {ep}: {why}; best was epoch {max(hist, key=lambda h: h[3])[0]}')
                    stopped = True
                    break
        if stopped:
            break
    # Continue from here with the chosen weights as a plain network (adapters already merged).
    final = ns['EEGNetCLIP'](CH, DIM, IN_SFREQ, dropout=DROPOUT_2, **ARCH_CFG)
    final.load_state_dict(best_state if KEEP == 'best' else last_state)
    model = final.to(DEVICE).eval()


## Step 4 — score it on held-out people

In [ ]:
if RUN_TRAINING:
    best_ep = max(hist, key=lambda h: h[3])
    vl_min = min(hist, key=lambda h: h[2])
    print(f'best val rank {best_ep[3]:.4f} at epoch {best_ep[0]} (KEEP = {KEEP!r}); '
          f'lowest val loss {vl_min[2]:.4f} at epoch {vl_min[0]}')

    pred = infer(Wte[0]).cpu().numpy()
    r_te = rank_pct(pred, Wte[1], emb)
    a_te, ch = top5(pred, Wte[1], emb)
    agg = {}
    for p_, y_, s_ in zip(pred, Wte[1], Wte[2]):
        agg.setdefault((s_, y_), []).append(p_)
    ks = sorted(agg)
    a_agg, _ = top5(np.stack([np.mean(agg[k_], 0) for k_ in ks]), np.array([k_[1] for k_ in ks]), emb)
    e = None
    print(f'\n  {"seen people, new images":<24} rank {r_te:.4f}   top-5 {a_te:5.2f}% (per-window)'
          f'   {a_agg:5.2f}% (subject-agg)   chance {ch:.2f}%')
    if HOLDOUT_PEOPLE:
        hm = np.isin(Wte[2], [str(p_) for p_ in HOLDOUT_PEOPLE])
        r_h = rank_pct(pred[hm], Wte[1][hm], emb)
        a_h, ch_h = top5(pred[hm], Wte[1][hm], emb)
        agg_h = {}
        for p_, y_, s_ in zip(pred[hm], Wte[1][hm], Wte[2][hm]):
            agg_h.setdefault((s_, y_), []).append(p_)
        ks_h = sorted(agg_h)
        a_hagg, _ = top5(np.stack([np.mean(agg_h[k_], 0) for k_ in ks_h]),
                         np.array([k_[1] for k_ in ks_h]), emb)
        print(f'\n  HELD-OUT {str(HOLDOUT_PEOPLE):<9} rank {r_h:.4f}   '
              f'top-5 {a_h:5.2f}%   {a_hagg:5.2f}% (subject-agg)   chance {ch_h:.2f}%   '
              f'{hm.sum():,} windows, {len(ks_h)} images - never fitted or validated')

    nullv = rank_null(infer(Xv).cpu().numpy(), yv)
    thr = float(np.percentile(nullv, 95))
    climb = best_ep[3] - hist[0][3]
    TRENDING = bool(best_ep[3] > thr and best_ep[0] > 1 and climb > 2 * nullv.std())
    print(f'\nval rank under permuted labels: mean {nullv.mean():.4f}  sd {nullv.std():.4f}  '
          f'95th {thr:.4f}; best {best_ep[3]:.4f} '
          f'({(best_ep[3] - nullv.mean()) / max(nullv.std(), 1e-9):.1f} sd above)')
    if TRENDING and e:
        verdict = 'beats' if a_te > e['top5'] else 'does not beat'
        print(f'Trending, and REVE {verdict} EEGNet on the same data '
              f'({a_te:.2f}% vs {e["top5"]:.2f}%).')
    elif TRENDING:
        print('Trending: it learns something that generalises to unseen images.')
    else:
        print('Not trending: validation rank never clears its own null.')

    torch.save(best_state if KEEP == 'best' else last_state, WEIGHTS / f'{RUN_TAG}_trainable.pt')
    print(f'saved -> {WEIGHTS / f"{RUN_TAG}_trainable.pt"} (adapters and head only)')


## Step 5 — package

Checks the plain network reloads through the submission file the way the harness does, and zips only if the checks pass.

In [ ]:
def package(model, tag, test_top5):
    """Write weights, config and solver; reload them the way the harness does; zip only if the reloaded
    submission reproduces the model it came from."""
    import importlib.util
    out = BASE / 'submissions' / f'track1_{tag}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    plain = to_plain(model).eval()
    torch.save({k: v.cpu() for k, v in plain.state_dict().items()}, out / 'weights.pt')
    (out / 'eegnet_config.json').write_text(json.dumps(dict(
        channels=CH, n_outputs=DIM, arch=ARCH_CFG, trained_on=f'Alljoined-1.6M ({tag})',
        test_top5=test_top5), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)

    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    def run(names, X):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                     'sfreq': IN_SFREQ, 'submission_dir': out,
                                     'device': DEVICE, 'n_outputs': DIM})
        return m.predict(torch.from_numpy(X.astype('float32'))).cpu().numpy()

    Xs = Wte[0][:8].astype('float32')
    base = run(CH, Xs)
    with torch.no_grad():
        model.eval()
        ref = model(torch.from_numpy(Xs).to(DEVICE)).float().cpu().numpy()
    perm = rng.permutation(len(CH))
    shuffled = run([CH[i] for i in perm], Xs[:, perm])
    wide = run(list(CH) + [f'X{i}' for i in range(31)],
               np.concatenate([Xs, rng.standard_normal((8, 31, Xs.shape[-1])).astype('float32')], 1))
    drop = list(CH[2:])
    part = run(drop, Xs[:, [CH.index(c) for c in drop]])
    checks = [
        ('reloaded submission = the trained model', np.allclose(base, ref, atol=1e-3),
         f'max |diff| {np.abs(base - ref).max():.1e}'),
        ('order-independent', np.allclose(base, shuffled, atol=1e-4),
         f'max |diff| {np.abs(base - shuffled).max():.1e}'),
        ('extra electrodes', np.allclose(base, wide, atol=1e-4), 'ignored, output identical'),
        ('missing electrodes', part.shape == base.shape and np.isfinite(part).all(),
         'zero-filled, still runs'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if not all(c[1] for c in checks):
        print('\nNot zipping: a check failed.')
        return None
    zpath = shutil.make_archive(str(out), 'zip', out)
    print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e6:.0f} MB)')
    return zpath


if RUN_TRAINING and MAKE_ZIP:
    if TRENDING or globals().get('FORCE_ZIP'):
        package(model, RUN_TAG, a_te)
    else:
        print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')
